# Build models to be simulated

## Load libraries

In [1]:
import libs.tools as tools

## Helper function for model manipulation

In [2]:
def set_number_of_arrivals(root, count: int, file: str):
    el = tools.get_element(root, 'ModelClients', file)
    el.text = str(count)


def set_operator_count(root, count: int, file: str):
    el1 = tools.get_element(root, 'Resources', file)
    el2 = tools.get_element(el1, 'Resource', file)
    el2.attrib["Value"] = str(count)


def set_service_batch_size(root, min: int, max: int, file: str):
    el1 = tools.get_element(root, 'ModelElements', file)
    el2 = tools.get_element(el1, 'ModelElementProcessStation', file)
    el3 = tools.get_or_add_element(el2, 'ModelElementBatchData')
    el3.attrib["Minimum"] = str(min)
    el3.attrib["Maximum"] = str(max)


def set_service_time(root, dist: str, file: str):
    el1 = tools.get_element(root, 'ModelElements', file)
    el2 = tools.get_element(el1, 'ModelElementProcessStation', file)
    el3 = tools.get_element(el2, 'ModelElementDistribution', file)
    el3.text = dist


def set_service_time_exp(root, mean: float, file: str):
    set_service_time(root, "Exponential distribution (" + str(mean) + ")", file)


def set_service_time_log(root, mean: float, std: float, file: str):
    set_service_time(root, "Lognormal distribution (" + str(mean) + ";" + str(std) + ")", file)


def set_inter_arrival(root, mean: float, b: int | list[int], file: str):
    el1 = tools.get_element(root, 'ModelElements', file)
    el2 = tools.get_element(el1, 'ModelElementSource', file)
    el3 = tools.get_element(el2, 'ModelElementDistribution', file)
    el3.text = f'Exponential distribution ({mean})'
    el3 = tools.get_element(el2, 'ModelElementBatchData', file)
    if isinstance(b, int):
        b = [b]
    el3.attrib["Size"] = ";".join([str(x) for x in b])


def set_variable(root, name: str, value: str, file: str):
    els = tools.get_elements(root, 'InitialVariable', file)
    for el in els:
        if el.attrib["Name"] == name:
            el.text = value
            return
    raise ValueError(f"Failed to find variable '{name}' in XML file: {file}")

## Load base model

In [3]:
base_file = "Models_Base/Batch.xml"
base_file_time_last = "Models_Base/BatchWithTimedReleaseLast.xml"
base_file_time_max = "Models_Base/BatchWithTimedReleaseMax.xml"
base_file_time_last12 = "Models_Base/BatchWithTimedReleaseLast12.xml"
base_file_time_max12 = "Models_Base/BatchWithTimedReleaseMax12.xml"

tree1, root1 = tools.load_xml_file(base_file)
tree2, root2 = tools.load_xml_file(base_file_time_last)
tree3, root3 = tools.load_xml_file(base_file_time_max)
tree4, root4 = tools.load_xml_file(base_file_time_last12)
tree5, root5 = tools.load_xml_file(base_file_time_max12)

## Base setup for all models

In [4]:
arrival_count = int(1E10)

c_needed = 16  # Needs to be a power of 2

# rho levels for all models
rho_levels = [0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95]

# rates for bI for the models with batch arrival
bI_rates_levels = [
    [1, 1, 1, 1],
    [1, 2, 3, 4],
    [4, 3, 2, 1],
    [1, 2, 2, 1],
    [2, 1, 1, 2]
]

# maximum waiting time (in seconds) of the *last* (newest) customer (models 6, 7, 10, 11)
release_times = [15, 30, 45, 60, 75, 90, 105, 120, 135, 150, 165, 180, 195, 210, 225, 240, 255, 270, 285, 300]

# maximum waiting time (in minutes) of the *first* (oldest) customer in queue (models 8, 9, 12, 13)
max_waiting_minutes = [15, 18, 21, 24, 27, 30]

In [5]:
set_number_of_arrivals(root1, arrival_count, base_file)
set_number_of_arrivals(root2, arrival_count, base_file_time_last)
set_number_of_arrivals(root3, arrival_count, base_file_time_max)
set_number_of_arrivals(root4, arrival_count, base_file_time_last12)
set_number_of_arrivals(root5, arrival_count, base_file_time_max12)

## Generate models 1

### S=Exp, CV[S]=1 and fixed batch size

In [6]:
for rho in rho_levels:

    mean = 100 * rho * c_needed
    set_service_time_exp(root1, mean, base_file)

    c = c_needed
    while c >= 1:
        b = int(c_needed / c)

        set_operator_count(root1, c, base_file)
        set_service_batch_size(root1, b, b, base_file)

        tools.save_xml_file(tree1, f"Models_Generated/Batch-exp-rho{int(rho * 100)}-b{b}.xml")

        c = int(c / 2)

### S=Log-Normal, CV[S]$\in\{0.5,1,1.5\}$ and fixed batch size

In [7]:
for CVS in [0.5,1,1.5]:
    for rho in rho_levels:

        mean = 100 * rho * c_needed
        set_service_time_log(root1, mean, mean * CVS, base_file)

        c = c_needed
        while c >= 1:
            b = int(c_needed / c)

            set_operator_count(root1, c, base_file)
            set_service_batch_size(root1, b, b, base_file)

            tools.save_xml_file(tree1, f"Models_Generated/Batch-log-rho{int(rho * 100)}-CV{CVS}-b{b}.xml")

            c = int(c / 2)

## Generate models 1a

### S=Exp, CV[S]=1 and fixed batch size (needed as base model for comparison)

In [ ]:
for bI_rates in bI_rates_levels:
    bI_rates_str = "".join([str(bI) for bI in bI_rates])
    i_means = [100, 200, 300, 400]
    i_mean = sum([bI_rates[i] * i_means[i] for i in range(len(bI_rates))]) / sum(bI_rates)

    for rho in rho_levels:

        mean = 100 * rho * c_needed
        set_service_time_exp(root1, mean, base_file)

        c = 1
        b = c_needed

        set_operator_count(root1, c, base_file)
        set_service_batch_size(root1, b, b, base_file)
        set_inter_arrival(root1, i_mean, bI_rates, base_file)

        tools.save_xml_file(tree1, f"Models_Generated/Batch-exp-rho{int(rho * 100)}-b{b}-bIRange{bI_rates_str}.xml")

### S=Log-Normal, CV[S]$\in\{0.5,1,1.5\}$ and fixed batch size (needed as base model for comparison)

In [7]:
for bI_rates in bI_rates_levels:
    bI_rates_str = "".join([str(bI) for bI in bI_rates])
    i_means = [100, 200, 300, 400]
    i_mean = sum([bI_rates[i] * i_means[i] for i in range(len(bI_rates))]) / sum(bI_rates)

    for CVS in [0.5, 1, 1.5]:
        for rho in rho_levels:

            mean = 100 * rho * c_needed
            set_service_time_log(root1, mean, mean * CVS, base_file)

            c = 1
            b = c_needed

            set_operator_count(root1, c, base_file)
            set_service_batch_size(root1, b, b, base_file)
            set_inter_arrival(root1, i_mean, bI_rates, base_file)

            tools.save_xml_file(tree1, f"Models_Generated/Batch-log-rho{int(rho * 100)}-CV{CVS}-b{b}-bIRange{bI_rates_str}.xml")

## Generate models 2

### S=Exp, CV[S]=1 and batch arrival (2...4); service batch size is here fixed to `c_needed`

In [8]:
for rho in rho_levels:

    mean = 100 * rho * c_needed
    set_service_time_exp(root1, mean, base_file)

    set_operator_count(root1, 1, base_file)
    set_service_batch_size(root1, c_needed, c_needed, base_file)

    for i in range(2, 5):
        set_inter_arrival(root1, 100 * i, i, base_file)

        tools.save_xml_file(tree1, f"Models_Generated/Batch-exp-rho{int(rho * 100)}-b{c_needed}-bI{i}.xml")

### S=Log-Normal, CV[S]$\in\{0.5,1,1.5\}$ and batch arrival (2...4); service batch size is here fixed to `c_needed`

In [9]:
for CVS in [0.5,1,1.5]:
    for rho in rho_levels:
        mean = 100 * rho * c_needed
        set_service_time_log(root1, mean, mean * CVS, base_file)

        set_operator_count(root1, 1, base_file)
        set_service_batch_size(root1, c_needed, c_needed, base_file)

        for i in range(2, 5):
            set_inter_arrival(root1, 100 * i, i, base_file)

            tools.save_xml_file(tree1, f"Models_Generated/Batch-log-rho{int(rho * 100)}-CV{CVS}-b{c_needed}-bI{i}.xml")

## Generate models 3

### S=Exp, CV[S]=1; service batch size is here fixed to $b_S\in\{8,4\}$

In [10]:
for c_needed_small in [8, 4]:
    for rho in rho_levels:
        set_inter_arrival(root1, 100 * (c_needed / c_needed_small), 1, base_file)

        mean = 100 * rho * c_needed  # We use the service time from bS=16
        set_service_time_exp(root1, mean, base_file)

        b = c_needed_small

        set_operator_count(root1, 1, base_file)
        set_service_batch_size(root1, b, b, base_file)

        tools.save_xml_file(tree1, f"Models_Generated/Compare-exp-rho{int(rho * 100)}-b{b}.xml")

### S=Log-Normal, CV[S]$\in\{0.5,1,1.5\}$; service batch size is here fixed to $b_S\in\{8,4\}$

In [11]:
for c_needed_small in [8, 4]:
    for CVS in [0.5, 1, 1.5]:
        for rho in rho_levels:
            set_inter_arrival(root1, 100 * (c_needed / c_needed_small), 1, base_file)

            mean = 100 * rho * c_needed  # We use the service time from bS=16
            set_service_time_log(root1, mean, mean * CVS, base_file)

            b = c_needed_small

            set_operator_count(root1, 1, base_file)
            set_service_batch_size(root1, b, b, base_file)

            tools.save_xml_file(tree1, f"Models_Generated/Compare-log-rho{int(rho * 100)}-CV{CVS}-b{b}.xml")

## Generate models 4

### S=Exp, CV[S]=1 and individual arrivals; service batch size is 12..16

In [12]:
for rho in rho_levels:
    mean = 100 * rho * c_needed
    set_service_time_exp(root1, mean, base_file)

    set_operator_count(root1, 1, base_file)

    for i in range(c_needed - 4, c_needed + 1):
        set_service_batch_size(root1, i, c_needed, base_file)
        tools.save_xml_file(tree1, f"Models_Generated/Range-exp-rho{int(rho * 100)}-b{c_needed}-bmin{i}.xml")

### S=Log-Normal, CV[S]$\in\{0.5,1,1.5\}$ and individual arrivals; service batch size is 12..16

In [13]:
for CVS in [0.5,1,1.5]:
    for rho in rho_levels:
        mean = 100 * rho * c_needed
        set_service_time_log(root1, mean, mean * CVS, base_file)

        set_operator_count(root1, 1, base_file)

        for i in range(c_needed - 4, c_needed + 1):
            set_service_batch_size(root1, i, c_needed, base_file)
            tools.save_xml_file(tree1, f"Models_Generated/Range-log-rho{int(rho * 100)}-CV{CVS}-b{c_needed}-bmin{i}.xml")

## Generate models 5

### S=Exp, CV[S]=1 and variable arrival batch size 1..4; service batch size is 12..16

In [14]:
for bI_rates in bI_rates_levels:

    bI_rates_str = "".join([str(bI) for bI in bI_rates])
    i_means = [100, 200, 300, 400]
    i_mean = sum([bI_rates[i] * i_means[i] for i in range(len(bI_rates))]) / sum(bI_rates)

    for rho in rho_levels:
        mean = 100 * rho * c_needed
        set_service_time_exp(root1, mean, base_file)

        set_operator_count(root1, 1, base_file)

        for i in range(c_needed - 4, c_needed + 1):
            set_service_batch_size(root1, i, c_needed, base_file)
            set_inter_arrival(root1, i_mean, bI_rates, base_file)

            tools.save_xml_file(tree1, f"Models_Generated/Range-exp-rho{int(rho * 100)}-b{c_needed}-bmin{i}-bIRange{bI_rates_str}.xml")

### S=Log-Normal, CV[S]$\in\{0.5,1,1.5\}$ and variable arrival batch size 1..4; service batch size is 12..16

In [15]:
for bI_rates in bI_rates_levels:

    bI_rates_str = "".join([str(bI) for bI in bI_rates])
    i_means = [100, 200, 300, 400]
    i_mean = sum([bI_rates[i] * i_means[i] for i in range(len(bI_rates))]) / sum(bI_rates)

    for CVS in [0.5,1,1.5]:
        for rho in rho_levels:
            mean = 100 * rho * c_needed
            set_service_time_log(root1, mean, mean * CVS, base_file)

            set_operator_count(root1, 1, base_file)

            for i in range(c_needed - 4, c_needed + 1):
                set_service_batch_size(root1, i, c_needed, base_file)
                set_inter_arrival(root1, i_mean, bI_rates, base_file)

                tools.save_xml_file(tree1, f"Models_Generated/Range-log-rho{int(rho * 100)}-CV{CVS}-b{c_needed}-bmin{i}-bIRange{bI_rates_str}.xml")

## Generate models 6

### S=Exp, CV[S]=1 and individual arrivals; service batch 16, but service also will start $\{15, 30, 45, 60, 75, 90, 105, 120\}$ sec. after last arrival

In [7]:
set_inter_arrival(root2, 100, 1, base_file_time_last)
set_operator_count(root2, 1, base_file_time_last)
set_service_batch_size(root2, 1, c_needed, base_file_time_last)

for rho in rho_levels:
    mean = 100 * rho * c_needed
    set_service_time_exp(root2, mean, base_file_time_last)

    for i in release_times:
        set_variable(root2, "BatchWaitingTime", str(i), base_file_time_last)

        tools.save_xml_file(tree2, f"Models_Generated/Time-exp-rho{int(rho * 100)}-b{c_needed}-bI1-time{i}.xml")

### S=Log-Normal, CV[S]$\in\{0.5,1,1.5\}$ and individual arrivals; service batch 16, but service also will start $\{15, 30, 45, 60, 75, 90, 105, 120\}$ sec. after last arrival

In [8]:
set_inter_arrival(root2, 100, 1, base_file_time_last)
set_operator_count(root2, 1, base_file_time_last)
set_service_batch_size(root2, 1, c_needed, base_file_time_last)

for CVS in [0.5,1,1.5]:
    for rho in rho_levels:
        mean = 100 * rho * c_needed
        set_service_time_log(root2, mean, mean * CVS, base_file_time_last)

        for i in release_times:
            set_variable(root2, "BatchWaitingTime", str(i), base_file_time_last)

            tools.save_xml_file(tree2, f"Models_Generated/Time-log-rho{int(rho * 100)}-CV{CVS}-b{c_needed}-bI1-time{i}.xml")

## Generate models 7

### S=Exp, CV[S]=1 and variable arrival batch size 1..4; service batch 16, but service also will start $\{15, 30, 45, 60, 75, 90, 105, 120\}$ sec. after last arrival

In [9]:
for bI_rates in bI_rates_levels:

    bI_rates_str = "".join([str(bI) for bI in bI_rates])
    i_means = [100, 200, 300, 400]
    i_mean = sum([bI_rates[i] * i_means[i] for i in range(len(bI_rates))]) / sum(bI_rates)

    set_inter_arrival(root2, i_mean, bI_rates, base_file_time_last)
    set_operator_count(root2, 1, base_file_time_last)
    set_service_batch_size(root2, 1, c_needed, base_file_time_last)

    for rho in rho_levels:
        mean = 100 * rho * c_needed
        set_service_time_exp(root2, mean, base_file_time_last)

        for i in release_times:
            set_variable(root2, "BatchWaitingTime", str(i), base_file_time_last)

            tools.save_xml_file(tree2, f"Models_Generated/Time-exp-rho{int(rho * 100)}-b{c_needed}-bIRange{bI_rates_str}-time{i}.xml")

### S=Log-Normal, CV[S]$\in\{0.5,1,1.5\}$ and variable arrival batch size 1..4; service batch 16, but service also will start $\{15, 30, 45, 60, 75, 90, 105, 120\}$ sec. after last arrival

In [10]:
for bI_rates in bI_rates_levels:

    bI_rates_str = "".join([str(bI) for bI in bI_rates])
    i_means = [100, 200, 300, 400]
    i_mean = sum([bI_rates[i] * i_means[i] for i in range(len(bI_rates))]) / sum(bI_rates)

    set_inter_arrival(root2, i_mean, bI_rates, base_file_time_last)
    set_operator_count(root2, 1, base_file_time_last)
    set_service_batch_size(root2, 1, c_needed, base_file_time_last)

    for CVS in [0.5,1,1.5]:
        for rho in rho_levels:
            mean = 100 * rho * c_needed
            set_service_time_log(root2, mean, mean * CVS, base_file_time_last)

            for i in release_times:
                set_variable(root2, "BatchWaitingTime", str(i), base_file_time_last)

                tools.save_xml_file(tree2, f"Models_Generated/Time-log-rho{int(rho * 100)}-CV{CVS}-b{c_needed}-bIRange{bI_rates_str}-time{i}.xml")

## Generate models 8

### S=Exp, CV[S]=1 and individual arrivals; service batch 16, but service also will start at max waiting time $\{15, 18, 21, 24, 27, 30\}$ min.

In [16]:
set_inter_arrival(root3, 100, 1, base_file_time_max)
set_operator_count(root3, 1, base_file_time_max)
set_service_batch_size(root3, 1, c_needed, base_file_time_max)

for rho in rho_levels:
    mean = 100 * rho * c_needed
    set_service_time_exp(root3, mean, base_file_time_max)

    for i in max_waiting_minutes:
        set_variable(root3, "BatchWaitingTime", str(i * 60), base_file_time_max)

        tools.save_xml_file(tree3, f"Models_Generated/TimeMax-exp-rho{int(rho * 100)}-b{c_needed}-bI1-time{i*60}.xml")

### S=Log-Normal, CV[S]$\in\{0.5,1,1.5\}$ and individual arrivals; service batch 16, but service also will start at max waiting time $\{15, 18, 21, 24, 27, 30\}$ min.

In [17]:
set_inter_arrival(root3, 100, 1, base_file_time_max)
set_operator_count(root3, 1, base_file_time_max)
set_service_batch_size(root3, 1, c_needed, base_file_time_max)

for CVS in [0.5, 1, 1.5]:
    for rho in rho_levels:
        mean = 100 * rho * c_needed
        set_service_time_log(root3, mean, mean * CVS, base_file_time_max)

        for i in max_waiting_minutes:
            set_variable(root3, "BatchWaitingTime", str(i * 60), base_file_time_max)

            tools.save_xml_file(tree3, f"Models_Generated/TimeMax-log-rho{int(rho * 100)}-CV{CVS}-b{c_needed}-bI1-time{i*60}.xml")

## Generate models 9

### S=Exp, CV[S]=1 and variable arrival batch size 1..4; service batch 16, but service also will start at max waiting time $\{15, 18, 21, 24, 27, 30\}$ min.

In [6]:
for bI_rates in bI_rates_levels:

    bI_rates_str = "".join([str(bI) for bI in bI_rates])
    i_means = [100, 200, 300, 400]
    i_mean = sum([bI_rates[i] * i_means[i] for i in range(len(bI_rates))]) / sum(bI_rates)

    set_inter_arrival(root3, i_mean, bI_rates, base_file_time_max)
    set_operator_count(root3, 1, base_file_time_max)
    set_service_batch_size(root3, 1, c_needed, base_file_time_max)

    for rho in rho_levels:
        mean = 100 * rho * c_needed
        set_service_time_exp(root3, mean, base_file_time_max)

        for i in max_waiting_minutes:
            set_variable(root3, "BatchWaitingTime", str(i * 60), base_file_time_max)

            tools.save_xml_file(tree3, f"Models_Generated/TimeMax-exp-rho{int(rho * 100)}-b{c_needed}-bIRange{bI_rates_str}-time{i*60}.xml")

### S=Log-Normal, CV[S]$\in\{0.5,1,1.5\}$ and variable arrival batch size 1..4; service batch 16, but service also will start at max waiting time $\{15, 18, 21, 24, 27, 30\}$ min.

In [6]:
for bI_rates in bI_rates_levels:

    bI_rates_str = "".join([str(bI) for bI in bI_rates])
    i_means = [100, 200, 300, 400]
    i_mean = sum([bI_rates[i] * i_means[i] for i in range(len(bI_rates))]) / sum(bI_rates)

    set_inter_arrival(root3, i_mean, bI_rates, base_file_time_max)
    set_operator_count(root3, 1, base_file_time_max)
    set_service_batch_size(root3, 1, c_needed, base_file_time_max)

    for CVS in [0.5, 1, 1.5]:
        for rho in rho_levels:
            mean = 100 * rho * c_needed
            set_service_time_log(root3, mean, mean * CVS, base_file_time_max)

            for i in max_waiting_minutes:
                set_variable(root3, "BatchWaitingTime", str(i * 60), base_file_time_max)

                tools.save_xml_file(tree3, f"Models_Generated/TimeMax-log-rho{int(rho * 100)}-CV{CVS}-b{c_needed}-bIRange{bI_rates_str}-time{i*60}.xml")

## Generate models 10

### S=Exp, CV[S]=1 and individual arrivals; service batch 12..16, but service also will start $\{15, 30, 45, 60, 75, 90, 105, 120\}$ sec. after last arrival

In [8]:
set_inter_arrival(root4, 100, 1, base_file_time_last12)
set_operator_count(root4, 1, base_file_time_last12)
set_service_batch_size(root4, 1, c_needed, base_file_time_last12)

for rho in rho_levels:
    mean = 100 * rho * c_needed
    set_service_time_exp(root4, mean, base_file_time_last12)

    for i in release_times:
        set_variable(root4, "BatchWaitingTime", str(i), base_file_time_last12)

        tools.save_xml_file(tree4, f"Models_Generated/Time-exp-rho{int(rho * 100)}-b{c_needed}-bmin12-bI1-time{i}.xml")

### S=Log-Normal, CV[S]$\in\{0.5,1,1.5\}$ and individual arrivals; service batch 12..16, but service also will start $\{15, 30, 45, 60, 75, 90, 105, 120\}$ sec. after last arrival

In [9]:
set_inter_arrival(root4, 100, 1, base_file_time_last12)
set_operator_count(root4, 1, base_file_time_last12)
set_service_batch_size(root4, 1, c_needed, base_file_time_last12)

for CVS in [0.5,1,1.5]:
    for rho in rho_levels:
        mean = 100 * rho * c_needed
        set_service_time_log(root4, mean, mean * CVS, base_file_time_last12)

        for i in release_times:
            set_variable(root4, "BatchWaitingTime", str(i), base_file_time_last12)

            tools.save_xml_file(tree4, f"Models_Generated/Time-log-rho{int(rho * 100)}-CV{CVS}-b{c_needed}-bmin12-bI1-time{i}.xml")

## Generate models 11

### S=Exp, CV[S]=1 and variable arrival batch size 1..4; service batch 12..16, but service also will start $\{15, 30, 45, 60, 75, 90, 105, 120\}$ sec. after last arrival

In [6]:
for bI_rates in bI_rates_levels:

    bI_rates_str = "".join([str(bI) for bI in bI_rates])
    i_means = [100, 200, 300, 400]
    i_mean = sum([bI_rates[i] * i_means[i] for i in range(len(bI_rates))]) / sum(bI_rates)

    set_inter_arrival(root4, i_mean, bI_rates, base_file_time_last12)
    set_operator_count(root4, 1, base_file_time_last12)
    set_service_batch_size(root4, 1, c_needed, base_file_time_last12)

    for rho in rho_levels:
        mean = 100 * rho * c_needed
        set_service_time_exp(root4, mean, base_file_time_last12)

        for i in release_times:
            set_variable(root4, "BatchWaitingTime", str(i), base_file_time_last12)

            tools.save_xml_file(tree4, f"Models_Generated/Time-exp-rho{int(rho * 100)}-b{c_needed}-bmin12-bIRange{bI_rates_str}-time{i}.xml")

### S=Log-Normal, CV[S]$\in\{0.5,1,1.5\}$ and variable arrival batch size 1..4; service batch 12..16, but service also will start $\{15, 30, 45, 60, 75, 90, 105, 120\}$ sec. after last arrival

In [7]:
for bI_rates in bI_rates_levels:

    bI_rates_str = "".join([str(bI) for bI in bI_rates])
    i_means = [100, 200, 300, 400]
    i_mean = sum([bI_rates[i] * i_means[i] for i in range(len(bI_rates))]) / sum(bI_rates)

    set_inter_arrival(root4, i_mean, bI_rates, base_file_time_last12)
    set_operator_count(root4, 1, base_file_time_last12)
    set_service_batch_size(root4, 1, c_needed, base_file_time_last12)

    for CVS in [0.5,1,1.5]:
        for rho in rho_levels:
            mean = 100 * rho * c_needed
            set_service_time_log(root4, mean, mean * CVS, base_file_time_last12)

            for i in release_times:
                set_variable(root4, "BatchWaitingTime", str(i), base_file_time_last12)

                tools.save_xml_file(tree4, f"Models_Generated/Time-log-rho{int(rho * 100)}-CV{CVS}-b{c_needed}-bmin12-bIRange{bI_rates_str}-time{i}.xml")

## Generate models 12

### S=Exp, CV[S]=1 and individual arrivals; service batch 12..16, but service also will start at max waiting time $\{15, 18, 21, 24, 27, 30\}$ min.

In [6]:
set_inter_arrival(root5, 100, 1, base_file_time_max12)
set_operator_count(root5, 1, base_file_time_max12)
set_service_batch_size(root5, 1, c_needed, base_file_time_max12)

for rho in rho_levels:
    mean = 100 * rho * c_needed
    set_service_time_exp(root5, mean, base_file_time_max12)

    for i in max_waiting_minutes:
        set_variable(root5, "BatchWaitingTime", str(i * 60), base_file_time_max12)

        tools.save_xml_file(tree5, f"Models_Generated/TimeMax-exp-rho{int(rho * 100)}-b{c_needed}-bmin12-bI1-time{i*60}.xml")

### S=Log-Normal, CV[S]$\in\{0.5,1,1.5\}$ and individual arrivals; service batch 12..16, but service also will start at max waiting time $\{15, 18, 21, 24, 27, 30\}$ min.

In [7]:
set_inter_arrival(root5, 100, 1, base_file_time_max12)
set_operator_count(root5, 1, base_file_time_max12)
set_service_batch_size(root5, 1, c_needed, base_file_time_max12)

for CVS in [0.5, 1, 1.5]:
    for rho in rho_levels:
        mean = 100 * rho * c_needed
        set_service_time_log(root5, mean, mean * CVS, base_file_time_max12)

        for i in max_waiting_minutes:
            set_variable(root5, "BatchWaitingTime", str(i * 60), base_file_time_max12)

            tools.save_xml_file(tree5, f"Models_Generated/TimeMax-log-rho{int(rho * 100)}-CV{CVS}-b{c_needed}-bmin12-bI1-time{i*60}.xml")

## Generate models 13

### S=Exp, CV[S]=1 and variable arrival batch size 1..4; service batch 12..16, but service also will start at max waiting time $\{15, 18, 21, 24, 27, 30\}$ min.

In [6]:
for bI_rates in bI_rates_levels:

    bI_rates_str = "".join([str(bI) for bI in bI_rates])
    i_means = [100, 200, 300, 400]
    i_mean = sum([bI_rates[i] * i_means[i] for i in range(len(bI_rates))]) / sum(bI_rates)

    set_inter_arrival(root5, i_mean, bI_rates, base_file_time_max12)
    set_operator_count(root5, 1, base_file_time_max12)
    set_service_batch_size(root5, 1, c_needed, base_file_time_max12)

    for rho in rho_levels:
        mean = 100 * rho * c_needed
        set_service_time_exp(root5, mean, base_file_time_max12)

        for i in max_waiting_minutes:
            set_variable(root5, "BatchWaitingTime", str(i * 60), base_file_time_max12)

            tools.save_xml_file(tree5, f"Models_Generated/TimeMax-exp-rho{int(rho * 100)}-b{c_needed}-bmin12-bIRange{bI_rates_str}-time{i*60}.xml")

### S=Log-Normal, CV[S]$\in\{0.5,1,1.5\}$ and variable arrival batch size 1..4; service batch 12..16, but service also will start at max waiting time $\{15, 18, 21, 24, 27, 30\}$ min.

In [6]:
for bI_rates in bI_rates_levels:

    bI_rates_str = "".join([str(bI) for bI in bI_rates])
    i_means = [100, 200, 300, 400]
    i_mean = sum([bI_rates[i] * i_means[i] for i in range(len(bI_rates))]) / sum(bI_rates)

    set_inter_arrival(root5, i_mean, bI_rates, base_file_time_max12)
    set_operator_count(root5, 1, base_file_time_max12)
    set_service_batch_size(root5, 1, c_needed, base_file_time_max12)

    for CVS in [0.5, 1, 1.5]:
        for rho in rho_levels:
            mean = 100 * rho * c_needed
            set_service_time_log(root5, mean, mean * CVS, base_file_time_max12)

            for i in max_waiting_minutes:
                set_variable(root5, "BatchWaitingTime", str(i * 60), base_file_time_max12)

                tools.save_xml_file(tree5, f"Models_Generated/TimeMax-log-rho{int(rho * 100)}-CV{CVS}-b{c_needed}-bmin12-bIRange{bI_rates_str}-time{i*60}.xml")